# Gemma-2-9B JuICE port smoke (FP16 wrap + tiny GPTQ)

Prove the Gemma-2 port on Kaggle **before** any `N_CALIB=128` quantize.

1. Full `google/gemma-2-9b` **base** eager FP16 wrap + one dual-run
2. Tiny GPTQ: 1 layer / 8 calib samples, `BACKEND.TORCH`, greedy decode **not garbage**

**Do not** use the tiny GPTQ folder for Stage 2. If decode is garbage or wrap cannot share `o_proj` with `TorchQuantLinear`, pivot to Qwen2.5-7B.

**~20–40 min** on GPU T4 ×2 (download + FP16 forward + 1-layer GPTQ).

## Before Run All

1. **GPU T4 ×2**, Internet **ON**, secret `HF_TOKEN`
2. Accept [google/gemma-2-9b](https://huggingface.co/google/gemma-2-9b) license (gated)
3. Attach **`kaggle_juice_gemma2_smoke.zip`** as a dataset
4. Save & Run All
5. Pass = config `256 / 16 / 8 / 50`, 42 wrapped layers, `head_output` `[B, S, 3584]`, GPTQ gens are real words



## 1 — Setup (gptqmodel 2.2.0 + transformers 4.51.3)


In [ ]:
import os, sys, subprocess, json, shutil, time
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = "/tmp/hf_home"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/hf_home"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# Same pin as Llama GPTQ: 4.56+/5.x decoder layers return a Tensor; gptqmodel 2.2
# then indexes [0] and drops batch. Kaggle ships transformers 5.x.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers==4.51.3",
        "accelerate>=0.26.0,<2.0",
        "safetensors>=0.4.0",
        "gptqmodel==2.2.0",
        "datasets>=2.14.0",
        "sentencepiece",
        "protobuf",
        "tqdm",
    ]
)

def _patch_no_init_weights() -> None:
    import transformers.modeling_utils as mu
    if hasattr(mu, "no_init_weights"):
        return
    try:
        from transformers.initialization import no_init_weights as _niw
        mu.no_init_weights = _niw
        print("patched no_init_weights from transformers.initialization")
        return
    except Exception:
        pass
    from contextlib import contextmanager

    @contextmanager
    def _noop(_enable=True):
        yield

    mu.no_init_weights = _noop
    print("patched no_init_weights with noop shim")

_patch_no_init_weights()

import torch
import transformers
import gptqmodel

print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "ngpu", torch.cuda.device_count())
print("transformers", transformers.__version__, "gptqmodel", gptqmodel.__version__)
assert torch.cuda.is_available(), "Need GPU T4 x2"
assert transformers.__version__.startswith("4.51."), transformers.__version__
if torch.cuda.device_count() < 2:
    print("WARNING: prefer T4 x2 — Gemma-2-9B FP16 is ~18GB")

try:
    from transformers.models.gemma2.modeling_gemma2 import Gemma2DecoderLayer
    _orig = Gemma2DecoderLayer.forward

    def _gemma2_layer_fwd_tuple(self, *args, **kwargs):
        out = _orig(self, *args, **kwargs)
        if torch.is_tensor(out):
            return (out,)
        return out

    Gemma2DecoderLayer.forward = _gemma2_layer_fwd_tuple
    print("Gemma2DecoderLayer.forward patched to always return a tuple")
except ImportError:
    print("no Gemma2DecoderLayer (unexpected on transformers 4.51)")

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
login(token=UserSecretsClient().get_secret("HF_TOKEN"), add_to_git_credential=False)
print("HF login ok")



## 2 — Locate zip + GPU / disk


In [ ]:
def find_one(name: str) -> Path:
    hits = list(Path("/kaggle/working").rglob(name))
    if Path("/kaggle/input").exists():
        hits += list(Path("/kaggle/input").rglob(name))
    hits = [h for h in hits if h.is_file()]
    assert hits, f"Missing {name} — attach kaggle_juice_gemma2_smoke.zip as a dataset"
    hits.sort(key=lambda p: (0 if "gemma2" in str(p).lower() else 1, len(str(p))))
    print(name, "->", hits[0])
    return hits[0]

SMOKE = find_one("gemma2_juice_smoke.py")
ROOT = SMOKE.resolve().parents[1]
sys.path.insert(0, str(ROOT))
print("ROOT", ROOT)
print("data", (ROOT / "data" / "conflictqa_pilot250_popular.jsonl").is_file())

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU[{i}] {p.name} {p.total_memory/1024**3:.1f} GiB")

for label, path in [("working", "/kaggle/working"), ("tmp", "/tmp")]:
    total, used, free = shutil.disk_usage(path)
    print(f"disk {label}: free {free/1024**3:.1f} GiB / {total/1024**3:.1f} GiB")



## 3 — FP16 full 9B wrap + dual-run


In [ ]:
print("=== STAGE fp16 --full-weights ===")
t0 = time.time()
subprocess.check_call(
    [
        sys.executable,
        "-u",
        str(SMOKE),
        "--stage",
        "fp16",
        "--full-weights",
    ],
    cwd=str(ROOT),
    env={**os.environ, "PYTHONPATH": str(ROOT)},
)
print(f"=== STAGE fp16 done in {(time.time()-t0)/60:.1f} min ===")



## 4 — Tiny GPTQ (1 layer / 8 samples). Not a Stage 2 checkpoint.


In [ ]:
GPTQ_OUT = Path("/kaggle/working/gemma2_gptq_tiny_layer0")
print("=== STAGE gptq --full-weights ===")
t0 = time.time()
subprocess.check_call(
    [
        sys.executable,
        "-u",
        str(SMOKE),
        "--stage",
        "gptq",
        "--full-weights",
        "--gptq-out",
        str(GPTQ_OUT),
    ],
    cwd=str(ROOT),
    env={**os.environ, "PYTHONPATH": str(ROOT)},
)
print(f"=== STAGE gptq done in {(time.time()-t0)/60:.1f} min ===")



## 5 — Confirm, then stop. Do not N_CALIB=128 yet.


In [ ]:
meta = GPTQ_OUT / "gptq_tiny_smoke.json"
print("tiny gptq dir", GPTQ_OUT, "exists" if GPTQ_OUT.is_dir() else "MISSING")
print("gptq_tiny_smoke.json", meta.is_file())
if meta.is_file():
    print(meta.read_text(encoding="utf-8")[:4000])

print()
print("=" * 60)
print("PASS CHECKLIST")
print("  config: head_dim=256, heads=16, kv=8, softcap=50, hidden=3584, layers=42")
print("  wrap: GetHeadsGemma2Attention, 42 layers, head_output [B, S, 3584]")
print("  dual_run_logits: printed ALL OK after fp16")
print("  GPTQ gens: real words, not aaaa / empty")
print()
print("If GPTQ gens are garbage or wrap/o_proj fails: pivot to Qwen2.5-7B.")
print("If both passed: next is N_CALIB=128 with the Llama recipe (do not use this tiny ckpt).")
print("=" * 60)
print("SAVE VERSION NOW (optional — only the json/log matters; discard tiny GPTQ weights)")

